# Experiment: 5D cond 1D

dim(x)=4, dim(y)=1 — comparing LGD vs LGD-CM.

In [10]:
# ============================================================
# CONFIG — only this cell changes between notebooks
# ============================================================
EXPERIMENT_NAME   = "5D_cond_1D"
GLOBAL_SEED       = 42
FORCE_RETRAIN     = False

BASE_DIR          = "/content/conditional-matching-paper/simulations"
PARAMS_DIR        = f"{BASE_DIR}/params"
CHECKPOINT_DIR    = f"{BASE_DIR}/checkpoints/{EXPERIMENT_NAME}"
RESULTS_DIR       = f"{BASE_DIR}/results/{EXPERIMENT_NAME}"

# Architecture — Diffusion models
NBLOCKS           = 6
NUNITS            = 512

# Architecture — Consistency Model
NBLOCKS_CM        = 6
NUNITS_CM         = 512

# Training — Diffusion
NEPOCHS           = 40_000#20_000
BATCH_SIZE        = 4_096#512

# Training — Consistency Model
NEPOCHS_CM        = 40_000
BATCH_SIZE_CM     = 4_096

# Diffusion
DIFFUSION_STEPS   = 150

# Optimization
N_ATTEMP_OPTIM              = 25
NSAMPLES_IN_OPTIM_FOR_MMD   = 250
NUM_X_T_LGD                 = 3
NUM_X_T_LGD_CM              = 5

# GMM dimensions
CONDITION_ON      = 4   # dim(x)=4, dim(y)=1

In [2]:
!pip install flow_matching -q
!pip install POT -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.7/48.7 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 63.3 MB/s eta 0:00:00


In [11]:
import os, sys
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('HF')
if hf_token:
    login(token=hf_token)
else:
    login()

if 'google.colab' in str(get_ipython()):
    import getpass
    !pip install -q diffusers transformers accelerate xformers
    !pip install -q scikit-learn matplotlib Pillow

    github_token = userdata.get('GITHUB')
    token = github_token if github_token else getpass.getpass("Enter your GitHub personal access token: ")

    repo_url  = f"https://{token}@github.com/orineo1/conditional-matching-paper.git"
    repo_name = "conditional-matching-paper"
    branch    = "adding-simu-compare"

    if not os.path.exists(repo_name):
        !git clone {repo_url}
    else:
        print(f"Repo '{repo_name}' already cloned — pulling latest...")
        !cd {repo_name} && git pull

    !cd {repo_name} && git checkout {branch}

# ── point Python at simulations/src where all .py modules live ──
src_path = f"/content/{repo_name}/simulations/src"
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(f"Branch: {branch}")
print(f"src path on sys.path: {src_path}")

KeyboardInterrupt: 

In [12]:
import os, sys, time, json
import importlib
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
from functools import partial
from tqdm import trange

import Diffusion
import LossFunctions
import ConsistencyModels
import dist_utils
import Optimization
import experiment_utils
from ConsistencyModels import ConsistencyModeliCT
import evalModels

for mod in [Diffusion, LossFunctions, ConsistencyModels,
            dist_utils, Optimization, experiment_utils]:
    importlib.reload(mod)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR,    exist_ok=True)
os.makedirs(PARAMS_DIR,     exist_ok=True)
print("Imports done.")

Imports done.


In [13]:
env_info = experiment_utils.get_environment_info()
experiment_utils.print_environment_info(env_info)

ENVIRONMENT INFO
  timestamp: 2026-04-20T05:39:14.859363
  torch_version: 2.10.0+cu128
  cuda_available: True
  cuda_version: 12.8
  device_name: NVIDIA L4
  packages:
    torch: 2.10.0+cu128
    numpy: 2.0.2
    flow_matching: 1.0.10
    POT: 0.9.6.post1
    matplotlib: 3.10.0
    pandas: 2.2.2
    tqdm: 4.67.3


In [14]:
experiment_utils.set_global_seed(GLOBAL_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

[Seed] All random seeds set to 42
Using device: cuda


## GMM Parameters

In [15]:
# ============================================================
# GMM PARAMETERS
# Priority:
#   1. Load from PARAMS_DIR  (shared across runs, committed to repo)
#   2. Load from RESULTS_DIR (fallback from a previous run)
#   3. Generate fresh and save to both dirs
#
# To force regeneration: set FORCE_REGENERATE_PARAMS = True
# ============================================================
FORCE_REGENERATE_PARAMS = False

def _load_params():
    """Try PARAMS_DIR first, then RESULTS_DIR."""
    loaded = experiment_utils.load_gmm_params(PARAMS_DIR, EXPERIMENT_NAME)
    if loaded is not None:
        print(f"[GMM] Loaded from PARAMS_DIR: {PARAMS_DIR}")
        return loaded
    loaded = experiment_utils.load_gmm_params(RESULTS_DIR, EXPERIMENT_NAME)
    if loaded is not None:
        print(f"[GMM] Loaded from RESULTS_DIR: {RESULTS_DIR}")
        return loaded
    return None

loaded = None if FORCE_REGENERATE_PARAMS else _load_params()

if loaded is not None:
    mu_list, Sigma_list, alpha, mog_means, mog_variances, weights, x_star = loaded
    mu_list    = [mu.float() for mu in mu_list]
    Sigma_list = [cov.float() for cov in Sigma_list]
    alpha      = alpha.float()
else:
    print("[GMM] Generating fresh parameters...")
    experiment_utils.set_global_seed(GLOBAL_SEED)   # seed generation for reproducibility
    mu_list, Sigma_list, alpha, mog_means, mog_variances, weights, x_star = \
        dist_utils.get_param_mog_with_target(
            dim_data=5, num_components=4, device='cpu',
            conditional_modes=2, distanceOrScale="Distance"
        )
    mog_means, mog_variances, weights = dist_utils.filter_and_normalize(
        mog_means, mog_variances, weights, threshold=0.001
    )
    mu_list    = [mu.float() for mu in mu_list]
    Sigma_list = [cov.float() for cov in Sigma_list]
    alpha      = alpha.float()

    # save to PARAMS_DIR (canonical, share across runs)
    experiment_utils.save_gmm_params(
        mu_list, Sigma_list, alpha,
        mog_means, mog_variances, weights, x_star,
        PARAMS_DIR, EXPERIMENT_NAME
    )


print(f"x_star = {x_star}")
print(f"Number of conditional modes after filtering: {len(mog_means)}")

[GMM] Parameters loaded from /content/conditional-matching-paper/simulations/params/5D_cond_1D_gmm_params.pt
[GMM] Loaded from PARAMS_DIR: /content/conditional-matching-paper/simulations/params
x_star = tensor([-4.4615, -0.2913, -0.9775, -4.8282])
Number of conditional modes after filtering: 2


## Data

In [16]:
experiment_utils.set_global_seed(GLOBAL_SEED)
X = dist_utils.generate_mog_samples(25_000, mu_list, Sigma_list, alpha).float().to(device)

[Seed] All random seeds set to 42


## Train Models

### Consistency Model — P(Y|X=x)

In [17]:
experiment_utils.set_global_seed(GLOBAL_SEED)

B, C      = X.shape
nfeatures = C - CONDITION_ON
data_generator_cm = partial(
    dist_utils.generate_mog_samples_not_differentiable,
    means=mu_list, variances=Sigma_list, weights=alpha
)

Cos_ConsistencyModeliCT = ConsistencyModeliCT(
    nfeatures=nfeatures, condition_on=CONDITION_ON,
    nunits=NUNITS_CM, depth=NBLOCKS_CM
)

_loaded_cm = experiment_utils.load_model_checkpoint(
    Cos_ConsistencyModeliCT, "CM", CHECKPOINT_DIR,
    EXPERIMENT_NAME, GLOBAL_SEED, device
) if not FORCE_RETRAIN else False

if not _loaded_cm:
    experiment_utils.set_global_seed(GLOBAL_SEED)
    Cos_ConsistencyModeliCT.train_model(
        X=None, nepochs=NEPOCHS_CM, batch_size=BATCH_SIZE_CM,
        device=device, condition=CONDITION_ON,
        data_generator=data_generator_cm, use_improved_training=True
    )
    experiment_utils.save_model_checkpoint(
        Cos_ConsistencyModeliCT, "CM", CHECKPOINT_DIR,
        EXPERIMENT_NAME, GLOBAL_SEED
    )

[Seed] All random seeds set to 42
[Checkpoint] CM loaded from /content/conditional-matching-paper/simulations/checkpoints/5D_cond_1D/5D_cond_1D_CM_seed42.pt


### Diffusion — P(Y|X=x)

In [18]:
experiment_utils.set_global_seed(GLOBAL_SEED)

X_train   = dist_utils.generate_mog_samples(1_000, mu_list, Sigma_list, alpha).float().to(device)
nfeatures = X_train.shape[1]

data_generator_diff_cond = partial(
    dist_utils.generate_mog_samples_not_differentiable,
    means=mu_list, variances=Sigma_list, weights=alpha, kernel_func=None
)

model_cond = Diffusion.DiffusionModel(
    nfeatures=nfeatures, nblocks=NBLOCKS, nunits=NUNITS,
    condition=True, condition_on=CONDITION_ON,
    diffusion_steps=DIFFUSION_STEPS
)

_loaded_diff_cond = experiment_utils.load_model_checkpoint(
    model_cond, "Diffusion_cond", CHECKPOINT_DIR,
    EXPERIMENT_NAME, GLOBAL_SEED, device
) if not FORCE_RETRAIN else False

if not _loaded_diff_cond:
    experiment_utils.set_global_seed(GLOBAL_SEED)
    model_cond.train_model(
        None, data_generator=data_generator_diff_cond,
        nepochs=NEPOCHS, batch_size=BATCH_SIZE,
        condition_on=CONDITION_ON
    )
    experiment_utils.save_model_checkpoint(
        model_cond, "Diffusion_cond", CHECKPOINT_DIR,
        EXPERIMENT_NAME, GLOBAL_SEED
    )

[Seed] All random seeds set to 42
[Checkpoint] No checkpoint found for Diffusion_cond at /content/conditional-matching-paper/simulations/checkpoints/5D_cond_1D/5D_cond_1D_Diffusion_cond_seed42.pt
[Seed] All random seeds set to 42


loss: 0.197655: 100%|██████████| 40000/40000 [3:18:08<00:00,  3.36it/s]

[Checkpoint] Diffusion_cond saved to /content/conditional-matching-paper/simulations/checkpoints/5D_cond_1D/5D_cond_1D_Diffusion_cond_seed42.pt


### Diffusion — P(X=x)

In [19]:
experiment_utils.set_global_seed(GLOBAL_SEED)

data_generator_diff_uncond = partial(
    dist_utils.generate_mog_samples_not_differentiable,
    means=mu_list, variances=Sigma_list, weights=alpha,
    kernel_func=lambda X: X[:, :CONDITION_ON]
)

model_uncond = Diffusion.DiffusionModel(
    nfeatures=CONDITION_ON, nblocks=NBLOCKS, nunits=NUNITS,
    condition=False, diffusion_steps=DIFFUSION_STEPS
)

_loaded_diff_uncond = experiment_utils.load_model_checkpoint(
    model_uncond, "Diffusion_uncond", CHECKPOINT_DIR,
    EXPERIMENT_NAME, GLOBAL_SEED, device
) if not FORCE_RETRAIN else False

if not _loaded_diff_uncond:
    experiment_utils.set_global_seed(GLOBAL_SEED)
    model_uncond.train_model(
        None, data_generator=data_generator_diff_uncond,
        nepochs=NEPOCHS, batch_size=BATCH_SIZE,
        condition_on=CONDITION_ON
    )
    experiment_utils.save_model_checkpoint(
        model_uncond, "Diffusion_uncond", CHECKPOINT_DIR,
        EXPERIMENT_NAME, GLOBAL_SEED
    )

[Seed] All random seeds set to 42
[Checkpoint] No checkpoint found for Diffusion_uncond at /content/conditional-matching-paper/simulations/checkpoints/5D_cond_1D/5D_cond_1D_Diffusion_uncond_seed42.pt
[Seed] All random seeds set to 42


loss: 0.687861: 100%|██████████| 40000/40000 [3:05:29<00:00,  3.59it/s]

[Checkpoint] Diffusion_uncond saved to /content/conditional-matching-paper/simulations/checkpoints/5D_cond_1D/5D_cond_1D_Diffusion_uncond_seed42.pt


## Optimize

### LGD

In [20]:
# NOTE: optimize_LGD call is untouched — only seed management added around it
best_x_t_LGD_list = []
l2_gmm_LGD_list   = []
l2_x_LGD_list     = []
lgd_times         = []
final_loss_LGD    = []

for i in trange(N_ATTEMP_OPTIM):
    run_seed = experiment_utils.set_run_seed(GLOBAL_SEED, i)

    start_time = time.time()
    best_x_t, best_x_0_cont_xt_hat, final_loss = Optimization.optimize_LGD(
        model_uncond, model_cond, mog_means, mog_variances, weights,
        mu_list, Sigma_list, alpha,
        nsamples=NSAMPLES_IN_OPTIM_FOR_MMD, loss="MMD", device=device,
        num_x_t=NUM_X_T_LGD
    )
    best_x_t = best_x_t.reshape(-1, 1)
    end_time = time.time()

    lgd_times.append(end_time - start_time)
    final_loss_LGD.append(final_loss)
    best_x_t_LGD_list.append(best_x_t)

    x_pred_t = best_x_t.float().view(-1).cpu()
    mu_pred, Sigma_pred = dist_utils.compute_conditionals(mu_list, Sigma_list, x_pred_t)
    w_pred = dist_utils.compute_alpha(mu_list, Sigma_list, alpha, x_pred_t)

    l2_gmm = dist_utils.gmm_l2_distance(
        mu_pred, Sigma_pred, w_pred, mog_means, mog_variances, weights
    )
    l2_x = (x_pred_t - x_star.float().cpu()).pow(2).sum().sqrt().item()

    l2_gmm_LGD_list.append(l2_gmm)
    l2_x_LGD_list.append(l2_x)
    print(f"[{i+1}] seed={run_seed} | L2 GMM: {l2_gmm:.6f} | L2 to x*: {l2_x:.6f}")

  0%|          | 0/25 [00:00<?, ?it/s]/content/conditional-matching-paper/simulations/src/dist_utils.py:466: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4480.)
  exponent = -0.5 * diff.T @ Sigma_22_inv @ diff
  4%|▍         | 1/25 [07:29<2:59:37, 449.08s/it]

[1] seed=42 | L2 GMM: 0.564978 | L2 to x*: 33.040230


  8%|▊         | 2/25 [14:57<2:52:01, 448.77s/it]

[2] seed=43 | L2 GMM: 1.142282 | L2 to x*: 15.464846


 12%|█▏        | 3/25 [22:30<2:45:13, 450.60s/it]

...

[23] seed=64 | L2 GMM: 1.162879 | L2 to x*: 25.338737


 96%|█████████▌| 24/25 [3:00:25<07:31, 451.69s/it]

[24] seed=65 | L2 GMM: 0.158530 | L2 to x*: 24.253519


100%|██████████| 25/25 [3:07:56<00:00, 451.05s/it]

[25] seed=66 | L2 GMM: 0.158279 | L2 to x*: 20.495754


### LGD-CM

In [21]:
# NOTE: optimize_LGD call is untouched — only seed management added around it
best_x_t_LGD_CM_list = []
l2_gmm_LGD_CM_list   = []
l2_x_LGD_CM_list     = []
lgd_cm_times         = []
final_loss_LGD_CM    = []

for i in trange(N_ATTEMP_OPTIM):
    run_seed = experiment_utils.set_run_seed(GLOBAL_SEED, i)

    start_time = time.time()
    best_x_t, best_x_0_cont_xt_hat, final_loss = Optimization.optimize_LGD(
        model_uncond, Cos_ConsistencyModeliCT,
        mog_means, mog_variances, weights,
        mu_list, Sigma_list, alpha,
        nsamples=NSAMPLES_IN_OPTIM_FOR_MMD, loss="MMD", device=device,
        CM=True, FLAG=False, num_x_t=NUM_X_T_LGD_CM
    )
    best_x_t = best_x_t.reshape(-1, 1)
    end_time = time.time()

    lgd_cm_times.append(end_time - start_time)
    final_loss_LGD_CM.append(final_loss)
    best_x_t_LGD_CM_list.append(best_x_t)

    x_pred_t = best_x_t.float().view(-1).cpu()
    mu_pred, Sigma_pred = dist_utils.compute_conditionals(mu_list, Sigma_list, x_pred_t)
    w_pred = dist_utils.compute_alpha(mu_list, Sigma_list, alpha, x_pred_t)

    l2_gmm = dist_utils.gmm_l2_distance(
        mu_pred, Sigma_pred, w_pred, mog_means, mog_variances, weights
    )
    l2_x = (x_pred_t - x_star.float().cpu()).pow(2).sum().sqrt().item()

    l2_gmm_LGD_CM_list.append(l2_gmm)
    l2_x_LGD_CM_list.append(l2_x)
    print(f"[{i+1}] seed={run_seed} | L2 GMM: {l2_gmm:.6f} | L2 to x*: {l2_x:.6f}")

  4%|▍         | 1/25 [00:35<14:03, 35.15s/it]

[1] seed=42 | L2 GMM: 0.157610 | L2 to x*: 31.787172


  8%|▊         | 2/25 [01:10<13:28, 35.14s/it]

[2] seed=43 | L2 GMM: 0.156921 | L2 to x*: 24.324697


 12%|█▏        | 3/25 [01:45<12:52, 35.11s/it]

...

[23] seed=64 | L2 GMM: 1.217804 | L2 to x*: 16.048468


 96%|█████████▌| 24/25 [14:02<00:35, 35.23s/it]

[24] seed=65 | L2 GMM: 0.156739 | L2 to x*: 23.521202


100%|██████████| 25/25 [14:38<00:00, 35.12s/it]

[25] seed=66 | L2 GMM: 0.156760 | L2 to x*: 23.165842


## Results

In [22]:
rows = [
    experiment_utils.summary_row("LGD",    l2_gmm_LGD_list,    l2_x_LGD_list,    lgd_times),
    experiment_utils.summary_row("LGD-CM", l2_gmm_LGD_CM_list, l2_x_LGD_CM_list, lgd_cm_times),
]
df = pd.DataFrame(rows).set_index("Method")
display(df)

rows_top10 = [
    experiment_utils.top10_stats("LGD",    final_loss_LGD,    l2_gmm_LGD_list,    l2_x_LGD_list,    lgd_times),
    experiment_utils.top10_stats("LGD-CM", final_loss_LGD_CM, l2_gmm_LGD_CM_list, l2_x_LGD_CM_list, lgd_cm_times),
]
df_top10 = pd.DataFrame(rows_top10).set_index("Method")
display(df_top10)

,L2 GMM mean,L2 GMM std,L2 to x* mean,L2 to x* std,Time mean (s),Time std (s)
Method,,,,,,
LGD,0.4855,0.3166,38.4658,46.7197,451.03,1.94
LGD-CM,0.4613,0.4317,25.3749,11.2605,35.11,0.13


,Loss mean,Loss std,L2 GMM mean,L2 GMM std,L2 to x* mean,L2 to x* std,Time mean (s),Time std (s),Top-k selected
Method,,,,,,,,,
LGD,0.1493,0.0423,0.4837,0.1189,33.1671,0.5286,451.94,2.12,10
LGD-CM,0.2956,0.0717,0.3304,0.3491,26.1795,9.3839,35.11,0.08,10


In [23]:
def to_python(val):
    if isinstance(val, torch.Tensor):
        return val.detach().cpu().tolist()
    if isinstance(val, np.ndarray):
        return val.tolist()
    if hasattr(val, "item"):
        return val.item()
    return val

results = {
    "experiment":  EXPERIMENT_NAME,
    "seed":        GLOBAL_SEED,
    "environment": env_info,
    "LGD": {
        "x_pred":     [to_python(x) for x in best_x_t_LGD_list],
        "final_loss": [to_python(l) for l in final_loss_LGD],
        "l2_gmm":     l2_gmm_LGD_list,
        "l2_x":       l2_x_LGD_list,
        "times":      lgd_times,
    },
    "LGD-CM": {
        "x_pred":     [to_python(x) for x in best_x_t_LGD_CM_list],
        "final_loss": [to_python(l) for l in final_loss_LGD_CM],
        "l2_gmm":     l2_gmm_LGD_CM_list,
        "l2_x":       l2_x_LGD_CM_list,
        "times":      lgd_cm_times,
    },
    "meta": {
        "n_attemp_optim":            N_ATTEMP_OPTIM,
        "nsamples_in_optim_for_mmd": NSAMPLES_IN_OPTIM_FOR_MMD,
        "x_star":                    to_python(x_star),
    },
}

path = os.path.join(RESULTS_DIR, f"{EXPERIMENT_NAME}_results_seed{GLOBAL_SEED}.json")
with open(path, "w") as f:
    json.dump(results, f, indent=2)
print(f"Results saved to {path}")

Results saved to /content/conditional-matching-paper/simulations/results/5D_cond_1D/5D_cond_1D_results_seed42.json
